# ChakraModel — ALL VARIANT WEIGHTS Evaluation
**Datasets:** gokulrocky/dataset-weights (variants) + gokulrocky/finalmuruga-harae (ViT-L baseline) + gokulrocky/chakramodel-evaluation-datasets (Kvasir/ClinicDB)

Per .antigravityrules.md:
- RULE 1: Every metric printed below came from real code on real data. No estimated numbers.
- RULE 2: Model load is verified 0 missing / 0 unexpected keys BEFORE any inference runs.
- RULE 5: Each model reports DONE / PARTIALLY DONE / NOT DONE.

**Models to evaluate:**
1. `chakra_transformer_best.pth` — ViT-L/16@384 baseline (from finalmuruga-harae)
2. `chakramodelweights/` — likely same or alternative ViT-L
3. `chakranet_focal_best.pth/` — Focal Loss + MC Dropout variant
4. `topo_chakranet_best.pth/` — Topology-aware (Persistent Homology) variant
5. `adabn_chakranet_best/` — Adaptive BN domain-adaptation variant
6. `fed_chakranet_global.pth/` — Federated global model

In [ ]:
# === CELL 1: Download all datasets ===
import kagglehub, os, glob

paths = {}
for key, slug in [
    ('variants',   'gokulrocky/dataset-weights'),
    ('baseline',   'gokulrocky/finalmuruga-harae'),
    ('eval_data',  'gokulrocky/chakramodel-evaluation-datasets'),
]:
    p = kagglehub.dataset_download(slug)
    paths[key] = p
    print(f'[OK] {key:12s} -> {p}')

# Show what's in variants dataset
VDIR = paths['variants']
print('\n=== Variant weight folders ===')
for entry in sorted(os.listdir(VDIR)):
    full = os.path.join(VDIR, entry)
    if os.path.isdir(full):
        files = []
        for root, dirs, fnames in os.walk(full):
            for fn in fnames:
                fp = os.path.join(root, fn)
                files.append((fn, os.path.getsize(fp)))
        print(f'  [{entry}/]')
        for fn, sz in sorted(files):
            print(f'    {fn}  ({sz/1e6:.1f} MB)')
    else:
        sz = os.path.getsize(full)
        print(f'  {entry}  ({sz/1e6:.1f} MB)')

In [ ]:
# === CELL 2: Load the ViT-L model class ===
import sys, torch, torch.nn as nn

WP = paths['baseline']
sys.path.insert(0, f'{WP}/src')
sys.path.insert(0, f'{WP}/src/chakra_transformer')

from chakra_transformer.transformer_segmenter import ChakraTransformerSegmenter

def load_checkpoint(ckpt_path, model_class, model_kwargs):
    """Load checkpoint with automatic module. prefix stripping.
    RULE 2: Prints missing/unexpected counts. ABORTS if mismatch > 0."""
    sd_raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(sd_raw, dict) and 'state_dict' in sd_raw:
        sd_raw = sd_raw['state_dict']
    elif isinstance(sd_raw, dict) and 'model' in sd_raw:
        sd_raw = sd_raw['model']

    model = model_class(**model_kwargs)

    # Try as-is first
    miss, unexp = model.load_state_dict(sd_raw, strict=False)
    if len(miss) == 0 and len(unexp) == 0:
        print(f'  [CLEAN LOAD] 0 missing, 0 unexpected — VERIFIED')
        return model

    # Try stripping module. prefix
    sd_stripped = {k.replace('module.', '', 1): v for k, v in sd_raw.items()}
    miss2, unexp2 = model.load_state_dict(sd_stripped, strict=False)
    if len(miss2) == 0 and len(unexp2) == 0:
        print(f'  [CLEAN LOAD after module. strip] 0 missing, 0 unexpected — VERIFIED')
        return model

    print(f'  [WARNING] missing={len(miss2)}, unexpected={len(unexp2)}')
    print(f'  First 5 missing: {miss2[:5]}')
    print(f'  First 5 unexpected: {unexp2[:5]}')
    print(f'  >>> RULE 2 VIOLATION: Cannot run inference with mismatched keys. Skipping this checkpoint. <<<')
    return None


def inspect_checkpoint(ckpt_path):
    """Print architecture fingerprint from key names, param count."""
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(sd, dict) and 'state_dict' in sd:
        sd = sd['state_dict']
    elif isinstance(sd, dict) and 'model' in sd:
        sd = sd['model']
    elif not isinstance(sd, dict):
        print(f'  Not a dict — type: {type(sd)}')
        return

    keys = list(sd.keys())
    total_params = sum(v.numel() for v in sd.values() if hasattr(v, 'numel'))
    print(f'  Total params in checkpoint: {total_params/1e6:.2f}M')
    print(f'  Total keys: {len(keys)}')
    print(f'  First 10 keys: {keys[:10]}')
    print(f'  Last 5 keys:   {keys[-5:]}')

    # Architecture fingerprint
    has_backbone_blocks = any('backbone.blocks' in k for k in keys)
    has_enc0 = any(k.startswith('enc0') or '.enc0' in k for k in keys)
    has_rfb = any('rfb' in k.lower() for k in keys)
    has_ra = any(k.startswith('ra') and k[2].isdigit() for k in keys)
    has_module_prefix = any(k.startswith('module.') for k in keys)
    print(f'  Fingerprint: ViT-L={has_backbone_blocks}, PraNet-CNN={has_enc0 or has_rfb}, module.prefix={has_module_prefix}')

print('Model loader utilities defined.')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

In [ ]:
# === CELL 3: Find all checkpoint files & inspect each ===
VDIR = paths['variants']
WP   = paths['baseline']

# Map: name -> path
ckpt_candidates = {}

# Baseline from finalmuruga-harae
for ext in ['*.pth', '*.pt']:
    for fp in glob.glob(os.path.join(WP, 'weights', ext)):
        ckpt_candidates[os.path.basename(fp)] = fp

# Variants from dataset-weights
for root, dirs, files in os.walk(VDIR):
    for fn in files:
        if fn.endswith(('.pth', '.pt', '.bin')):
            ckpt_candidates[fn] = os.path.join(root, fn)

print(f'Found {len(ckpt_candidates)} checkpoint(s):')
for name, path in sorted(ckpt_candidates.items()):
    sz = os.path.getsize(path)
    print(f'\n--- {name} ({sz/1e6:.1f} MB) ---')
    print(f'  Path: {path}')
    try:
        inspect_checkpoint(path)
    except Exception as e:
        print(f'  [INSPECT ERROR] {e}')

In [ ]:
# === CELL 4: Preprocessing + metrics (reused for all models) ===
import numpy as np, cv2, time
import torch.nn.functional as F

IMG_SIZE = 384
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def preprocess(img_bgr):
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
    img = img.astype(np.float32) / 255.0
    img = (img - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(img.transpose(2, 0, 1)).unsqueeze(0).cuda()

def dice_score(pred, gt, eps=1e-7):
    pred = (pred > 0.5).float()
    inter = (pred * gt).sum()
    return (2 * inter / (pred.sum() + gt.sum() + eps)).item()

def iou_score(pred, gt, eps=1e-7):
    pred = (pred > 0.5).float()
    inter = (pred * gt).sum()
    union = pred.sum() + gt.sum() - inter
    return (inter / (union + eps)).item()

def fbeta_score(pred, gt, beta2=0.3, eps=1e-7):
    pred_b = (pred > 0.5).float()
    tp = (pred_b * gt).sum()
    prec = tp / (pred_b.sum() + eps)
    rec  = tp / (gt.sum() + eps)
    return ((1 + beta2) * prec * rec / (beta2 * prec + rec + eps)).item()

print('Preprocessing and metric functions defined.')

In [ ]:
# === CELL 5: Build eval dataset loaders ===
from pathlib import Path

DP = paths['eval_data']

def build_pairs(dataset_name, use_last_10pct=False):
    """Return sorted list of (img_path, mask_path) tuples."""
    # Try multiple naming conventions
    candidates = [
        (Path(DP) / dataset_name / 'images', Path(DP) / dataset_name / 'masks'),
        (Path(DP) / dataset_name / 'images', Path(DP) / dataset_name / 'mask'),
        (Path(DP) / dataset_name / 'Original', Path(DP) / dataset_name / 'Ground Truth'),
        (Path(DP) / dataset_name / 'image', Path(DP) / dataset_name / 'mask'),
    ]
    img_dir, msk_dir = None, None
    for i_d, m_d in candidates:
        if i_d.exists() and m_d.exists():
            img_dir, msk_dir = i_d, m_d
            break
    if img_dir is None:
        # Search recursively
        base = Path(DP) / dataset_name
        imgs = sorted([p for ext in ['*.jpg','*.png','*.bmp'] for p in base.rglob(ext)])
        if imgs:
            img_dir = imgs[0].parent
        print(f'  [{dataset_name}] fallback scan: img_dir={img_dir}')
        return []

    imgs = sorted([p for ext in ['*.jpg','*.png','*.bmp'] for p in img_dir.glob(ext)])
    pairs = []
    for img_p in imgs:
        for ext in ['.png', '.jpg', '.bmp']:
            msk_p = msk_dir / (img_p.stem + ext)
            if msk_p.exists():
                pairs.append((str(img_p), str(msk_p)))
                break
    print(f'  [{dataset_name}] {len(imgs)} images -> {len(pairs)} paired')
    if use_last_10pct and pairs:
        cutoff = int(len(pairs) * 0.9)
        pairs = pairs[cutoff:]
        print(f'  [{dataset_name}] Using last 10%: {len(pairs)} pairs')
    return pairs

print('Building eval pairs...')
kvasir_pairs   = build_pairs('kvasir-seg', use_last_10pct=True)
clinicdb_pairs = build_pairs('cvc-clinicdb', use_last_10pct=False)
print(f'\nKvasir-SEG test set: {len(kvasir_pairs)} pairs')
print(f'CVC-ClinicDB:        {len(clinicdb_pairs)} pairs')

In [ ]:
# === CELL 6: Generic inference + evaluation function ===

@torch.no_grad()
def infer_single(model, img_bgr, use_amp=True):
    """Single-pass FP16 inference."""
    x = preprocess(img_bgr)
    with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
        logits = model(x)
    if logits.shape[-2:] != (img_bgr.shape[0], img_bgr.shape[1]):
        logits = F.interpolate(logits, size=(img_bgr.shape[0], img_bgr.shape[1]),
                               mode='bilinear', align_corners=False)
    return torch.sigmoid(logits.float())[0, 0]

@torch.no_grad()
def infer_tta(model_dp, img_bgr, use_amp=True):
    """4-way TTA: identity + hflip + vflip + both."""
    x = preprocess(img_bgr)
    xs = torch.cat([x, torch.flip(x,[3]), torch.flip(x,[2]), torch.flip(x,[2,3])], dim=0)
    with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
        ys = model_dp(xs)
    ys = torch.sigmoid(ys.float())
    ys = torch.stack([
        ys[0], torch.flip(ys[1],[2]), torch.flip(ys[2],[1]), torch.flip(ys[3],[1,2])
    ])
    prob = ys.mean(0)[0]  # [1, H, W]
    if prob.shape[-2:] != (img_bgr.shape[0], img_bgr.shape[1]):
        prob = F.interpolate(prob.unsqueeze(0), size=(img_bgr.shape[0], img_bgr.shape[1]),
                             mode='bilinear', align_corners=False)[0]
    return prob[0]


def eval_model_on_dataset(model, pairs, dataset_name, use_tta=False, model_dp=None):
    """Evaluate a model on list of (img,mask) pairs. Returns dict of metrics."""
    dices, ious, fbetas = [], [], []
    model.eval()
    for img_p, msk_p in pairs:
        img_bgr = cv2.imread(img_p)
        msk     = cv2.imread(msk_p, cv2.IMREAD_GRAYSCALE)
        if img_bgr is None or msk is None:
            continue
        gt = torch.from_numpy((msk > 127).astype(np.float32)).cuda()
        if use_tta and model_dp is not None:
            prob = infer_tta(model_dp, img_bgr)
        else:
            prob = infer_single(model, img_bgr)
        # Resize prob to match gt
        if prob.shape != gt.shape:
            prob = F.interpolate(prob.unsqueeze(0).unsqueeze(0),
                                 size=gt.shape, mode='bilinear', align_corners=False)[0,0]
        dices.append(dice_score(prob, gt))
        ious.append(iou_score(prob, gt))
        fbetas.append(fbeta_score(prob, gt))

    n = len(dices)
    res = {
        'dataset': dataset_name,
        'n': n,
        'dice':  round(float(np.mean(dices)), 4)  if dices else None,
        'iou':   round(float(np.mean(ious)), 4)   if ious  else None,
        'fbeta': round(float(np.mean(fbetas)), 4) if fbetas else None,
    }
    print(f'  {dataset_name} (n={n}): Dice={res["dice"]}  IoU={res["iou"]}  Fβ={res["fbeta"]}')
    return res


def bench_fps(model, batch_size=1, n_iter=30, warmup=10, use_amp=True):
    """FPS benchmark with cuda.synchronize fencing."""
    x = torch.randn(batch_size, 3, IMG_SIZE, IMG_SIZE, device='cuda')
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                model(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_iter):
            with torch.amp.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                model(x)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    fps = n_iter * batch_size / dt
    return round(fps, 2)

print('Inference and benchmark utilities defined.')

In [ ]:
# === CELL 7: EVALUATE ALL MODELS ===
# This cell tries to load and evaluate every checkpoint found.
# If a checkpoint fails the 0-missing/0-unexpected check, it is skipped (Rule 2).

import json
from pathlib import Path

ALL_RESULTS = []  # collect all model results

def try_load_vit_l(ckpt_path):
    """Try to load checkpoint as ViT-L/16@384 segmenter."""
    try:
        return load_checkpoint(
            ckpt_path,
            ChakraTransformerSegmenter,
            dict(backbone_name='vit_large_patch16_384', pretrained=False, num_classes=1)
        )
    except Exception as e:
        print(f'  ViT-L load failed: {e}')
        return None


def try_load_pranet(ckpt_path):
    """Try to load as PraNet-ResNet101 (combo1 architecture)."""
    try:
        # Import combo1 model from finalmuruga-harae src
        from chakra_transformer.combo1_model import PraNetResNet101  # adjust import if needed
        return load_checkpoint(ckpt_path, PraNetResNet101, {})
    except ImportError:
        # Try building it inline if the class isn't importable
        print('  combo1_model.py not importable, trying inline...')
        return None
    except Exception as e:
        print(f'  PraNet load failed: {e}')
        return None


# Process each checkpoint
for ckpt_name, ckpt_path in sorted(ckpt_candidates.items()):
    sz_mb = os.path.getsize(ckpt_path) / 1e6
    print(f'\n{'='*60}')
    print(f'MODEL: {ckpt_name}  ({sz_mb:.1f} MB)')
    print(f'PATH:  {ckpt_path}')
    print('='*60)

    # Step 1: Inspect
    try:
        inspect_checkpoint(ckpt_path)
    except Exception as e:
        print(f'  Inspect failed: {e}')
        continue

    # Step 2: Try loading
    model = try_load_vit_l(ckpt_path)
    arch_name = 'ViT-L/16@384'

    if model is None:
        print('  ViT-L failed. Trying PraNet-ResNet101...')
        model = try_load_pranet(ckpt_path)
        arch_name = 'PraNet-ResNet101'

    if model is None:
        print(f'  [SKIP] Could not load {ckpt_name} with any known architecture.')
        ALL_RESULTS.append({
            'model': ckpt_name, 'arch': 'UNKNOWN', 'size_mb': sz_mb,
            'status': 'LOAD_FAILED', 'kvasir': None, 'clinicdb': None, 'fps_fp16_bs1': None
        })
        continue

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Arch: {arch_name}, Params: {n_params:.2f}M')
    model = model.cuda().eval()
    model_dp = torch.nn.DataParallel(model, device_ids=list(range(torch.cuda.device_count())))

    # Step 3: FPS benchmark (single-pass FP16, bs=1)
    print('  Benchmarking FPS (bs=1, FP16, 30 iter, warmup=10)...')
    fps = bench_fps(model, batch_size=1, n_iter=30, warmup=10, use_amp=True)
    print(f'  FPS (single-GPU FP16 bs=1): {fps}')

    # Step 4: Evaluate on Kvasir-SEG (baseline, no TTA)
    print('  Evaluating on Kvasir-SEG (last 10%, baseline FP16)...')
    kv_base = eval_model_on_dataset(model, kvasir_pairs, 'Kvasir-SEG', use_tta=False)

    # Step 5: Evaluate on Kvasir-SEG (4-way TTA)
    print('  Evaluating on Kvasir-SEG (TTA)...')
    kv_tta = eval_model_on_dataset(model, kvasir_pairs, 'Kvasir-SEG+TTA', use_tta=True, model_dp=model_dp)

    # Step 6: Evaluate on CVC-ClinicDB (baseline)
    print('  Evaluating on CVC-ClinicDB (all 495, baseline FP16)...')
    cl_base = eval_model_on_dataset(model, clinicdb_pairs, 'CVC-ClinicDB', use_tta=False)

    # Step 7: Evaluate on CVC-ClinicDB (TTA)
    print('  Evaluating on CVC-ClinicDB (TTA)...')
    cl_tta = eval_model_on_dataset(model, clinicdb_pairs, 'CVC-ClinicDB+TTA', use_tta=True, model_dp=model_dp)

    ALL_RESULTS.append({
        'model': ckpt_name,
        'arch': arch_name,
        'size_mb': round(sz_mb, 1),
        'params_m': round(n_params, 2),
        'status': 'EVALUATED',
        'fps_fp16_bs1': fps,
        'kvasir_baseline_dice':   kv_base['dice'],
        'kvasir_tta_dice':        kv_tta['dice'],
        'clinicdb_baseline_dice': cl_base['dice'],
        'clinicdb_tta_dice':      cl_tta['dice'],
        'kvasir_tta_iou':         kv_tta['iou'],
        'clinicdb_tta_iou':       cl_tta['iou'],
    })

    # Free GPU memory before next model
    del model, model_dp
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('ALL MODELS PROCESSED')
print('='*60)
for r in ALL_RESULTS:
    print(json.dumps(r, indent=2))

In [ ]:
# === CELL 8: Comparison table — all variants vs SOTA ===
import csv, io

SOTA = [
    {'model': 'U-Net (2015)',      'kvasir_tta_dice': 0.818, 'clinicdb_tta_dice': 0.823, 'params_m': 31.0,  'fps_fp16_bs1': '~30'},
    {'model': 'U-Net++ (2018)',    'kvasir_tta_dice': 0.821, 'clinicdb_tta_dice': 0.794, 'params_m': 36.6,  'fps_fp16_bs1': '~30'},
    {'model': 'PraNet (2020)',     'kvasir_tta_dice': 0.898, 'clinicdb_tta_dice': 0.899, 'params_m': 32.6,  'fps_fp16_bs1': '~50'},
    {'model': 'SANet (2021)',      'kvasir_tta_dice': 0.904, 'clinicdb_tta_dice': 0.916, 'params_m': 23.9,  'fps_fp16_bs1': '~72'},
    {'model': 'Polyp-PVT (2021)', 'kvasir_tta_dice': 0.917, 'clinicdb_tta_dice': 0.937, 'params_m': 25.1,  'fps_fp16_bs1': '~40'},
    {'model': 'SSFormer-L (2022)','kvasir_tta_dice': 0.917, 'clinicdb_tta_dice': 0.906, 'params_m': 66.2,  'fps_fp16_bs1': '~28'},
    {'model': 'CFA-Net (2023)',   'kvasir_tta_dice': 0.923, 'clinicdb_tta_dice': 0.933, 'params_m': 30.1,  'fps_fp16_bs1': '~55'},
]

print('=' * 95)
print(f'{"Model":<35} {"Kvasir Dice":>12} {"ClinicDB Dice":>14} {"Params(M)":>10} {"FPS":>8}')
print('-' * 95)
for s in SOTA:
    print(f'{s["model"]:<35} {s["kvasir_tta_dice"]:>12.4f} {s["clinicdb_tta_dice"]:>14.4f} {s["params_m"]:>10.1f} {str(s["fps_fp16_bs1"]):>8}')
print('-' * 95)
for r in ALL_RESULTS:
    if r['status'] == 'EVALUATED':
        kv  = r.get('kvasir_tta_dice')   or 'N/A'
        cl  = r.get('clinicdb_tta_dice') or 'N/A'
        fps = r.get('fps_fp16_bs1')      or 'N/A'
        pm  = r.get('params_m')          or 'N/A'
        name = r['model'].replace('_best.pth','').replace('_best','')[:34]
        print(f'{name + " [OURS]":<35} {str(kv):>12} {str(cl):>14} {str(pm):>10} {str(fps):>8}')
    else:
        name = r['model'][:34]
        print(f'{name + " [FAILED]":<35} {"N/A":>12} {"N/A":>14} {str(r.get("size_mb","?"))+'MB':>10} {"N/A":>8}')
print('=' * 95)

# Save to CSV
os.makedirs('/kaggle/working/results', exist_ok=True)
outpath = '/kaggle/working/results/all_variants_comparison.csv'
fieldnames = ['model','arch','params_m','size_mb','status','fps_fp16_bs1',
              'kvasir_baseline_dice','kvasir_tta_dice','clinicdb_baseline_dice','clinicdb_tta_dice',
              'kvasir_tta_iou','clinicdb_tta_iou']
with open(outpath, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction='ignore')
    w.writeheader()
    w.writerows(ALL_RESULTS)
print(f'\nCSV saved: {outpath}')

In [ ]:
# === CELL 9: Write final markdown report ===

lines = []
lines.append('# ChakraModel — All Variant Weights Evaluation Report')
lines.append('**Hardware:** Kaggle T4 x2 GPU')
lines.append('**Per .antigravityrules.md Rule 1 & 2:** Only measured metrics below. Load verified 0 missing/unexpected before each run.')
lines.append('')
lines.append('## Model Results')
lines.append('')
lines.append('| Model | Arch | Params(M) | Kvasir Dice (TTA) | ClinicDB Dice (TTA) | FPS FP16 bs=1 | Status |')
lines.append('|-------|------|-----------|-------------------|---------------------|--------------|--------|')
for r in ALL_RESULTS:
    name = r['model'].replace('_best.pth','').replace('_best','')
    kv   = r.get('kvasir_tta_dice')   or 'N/A (load failed)'
    cl   = r.get('clinicdb_tta_dice') or 'N/A (load failed)'
    fps  = r.get('fps_fp16_bs1')      or 'N/A'
    pm   = r.get('params_m')          or '?'
    status = r.get('status', 'UNKNOWN')
    lines.append(f'| **{name}** | {r["arch"]} | {pm} | {kv} | {cl} | {fps} | {status} |')
lines.append('')
lines.append('## SOTA Comparison')
lines.append('')
lines.append('| Method | Kvasir Dice | ClinicDB Dice | Params(M) |')
lines.append('|--------|-------------|---------------|-----------|')
for s in SOTA:
    lines.append(f'| {s["model"]} | {s["kvasir_tta_dice"]} | {s["clinicdb_tta_dice"]} | {s["params_m"]} |')
lines.append('')
lines.append('## Dataset Paths Used')
lines.append('```')
lines.append(f'Variant weights: {paths["variants"]}')
lines.append(f'Baseline weights: {paths["baseline"]}/weights/')
lines.append(f'Eval data: {paths["eval_data"]}')
lines.append(f'Kvasir-SEG: last 10% of sorted filenames = {len(kvasir_pairs)} images')
lines.append(f'CVC-ClinicDB: all {len(clinicdb_pairs)} images (not standard 62-test split)')
lines.append('```')

report_md = '\n'.join(lines)
report_path = '/kaggle/working/results/all_variants_eval_report.md'
with open(report_path, 'w') as f:
    f.write(report_md)
print(report_md)
print(f'\nReport saved: {report_path}')